# Sentiment Analysis Model Comparison

## Project Overview
This project focuses on sentiment analysis of product reviews. We classify reviews into negative, neutral, and positive categories and compare multiple models.
The overall task is to predict whether each review is:
- `0` = negative
- `1` = neutral
- `2` = positive

## 1. Imports

The first step and most important step is to imports the main libraries used in the notebook.

Key libraries:
- `pandas` / `numpy`: data handling
- `torch`: deep learning backend
- `sklearn`: train/test split and evaluation metrics
- `transformers`: Hugging Face BERT model and tokenizer
- `datasets`: converts data into Hugging Face dataset format

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Install / update required packages

This installs or updates the Hugging Face training packages.

In [2]:
import sys
!{sys.executable} -m pip install -U "transformers[torch]" "accelerate>=1.1.0"

## Google Drive setup 

This code is commented out. They were probably used when running the notebook in Google Colab.

If you run locally, you can ignore these cells as long as `AllProductReviews.csv` is in the same folder as the notebook.


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# cd /content/drive/MyDrive/NLP/Assignment3/dataset

/content/drive/MyDrive/NLP/Assignment3/dataset


## Load and clean the dataset

In this step we read the CSV file, renames the useful columns, keeps only the review text and rating, then removes missing values.

Original columns used:
- `ReviewBody` → renamed to `reviewText`
- `ReviewStar` → renamed to `overall`

At this stage, the dataset still has star ratings, not sentiment labels yet.

In [3]:
df = pd.read_csv("AllProductReviews.csv")
print(f"Raw Dataset: \n{df.head(1)}")

df = df.rename(columns={
    "ReviewBody": "reviewText",
    "ReviewStar": "overall"
})

# Keep only the needed column
df = df[["reviewText", "overall"]]

# Remove the missing values (if exists)
print("")
print(df[["reviewText", "overall"]].isnull().sum())
df = df.dropna(subset=["reviewText", "overall"])

print("")
print(f"Dataset: \n{df.head(1)}")

FileNotFoundError: [Errno 2] No such file or directory: 'AllProductReviews.csv'

## Convert star ratings into sentiment labels

This step converts the original 1–5 star ratings into 3 sentiment classes:

- 1 or 2 stars → `0` = negative
- 3 stars → `1` = neutral
- 4 or 5 stars → `2` = positive

This makes the task a **3-class classification problem**.


In [6]:
def convert_rating_to_label(rating):  # Convert the rating to Negative, Neutral, and Positive
    if rating in [1, 2]:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

## Apply the sentiment label conversion

This creates a new column called `label`, which is what the model will learn to predict.

In [7]:
df["label"] = df["overall"].apply(convert_rating_to_label)
print(df.head(2))

                                          reviewText  overall  label
0  No doubt it has a great bass and to a great ex...        3      1
1  This  earphones are unreliable, i bought it be...        1      0


## 7. Check class distribution

This counts how many negative, neutral, and positive examples are in the dataset.

This is important because if one class is much larger than the others, the model may become biased toward that class.

In [8]:
label_names = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label_counts = df["label"].value_counts().sort_index()
label_counts_named = label_counts.rename(index=label_names)

print(label_counts_named)

label
negative    3432
neutral     1503
positive    9402
Name: count, dtype: int64


## Split data into train, validation, and test sets

This creates three datasets:

- **Training set**: used to train the model
- **Validation set**: used to tune/check the model during development
- **Test set**: used at the end to measure final performance

The split is stratified, meaning each split should keep roughly the same proportion of negative, neutral, and positive reviews. This is is also kept the same across all the methods being tested to ensure we can evaluate methods fairly 

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df["reviewText"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("X_train size:", len(X_train))
print("X_test size:", len(X_test))
print("X_val size:", len(X_val))

X_train size: 11469
X_test size: 1434
X_val size: 1434


## Method 1: BERT
### Convert pandas data into Hugging Face Dataset format

BERT training with `Trainer` expects data in Hugging Face `Dataset` format.

This code converts the train, validation, and test splits into that format.

In [10]:
# Converts a pandas into a normal Python list

train_set = Dataset.from_dict(
    {
        "reviewText": X_train.to_list(),
        "label": y_train.to_list()
    }
)

test_set = Dataset.from_dict(
    {
        "reviewText": X_test.to_list(),
        "label": y_test.to_list()
    }
)

val_set = Dataset.from_dict(
    {
        "reviewText": X_val.to_list(),
        "label": y_val.to_list()
    }
)

### Load BERT tokenizer

This uses `bert-base-uncased`, which means:
- BERT is pretrained
- text is lowercased
- words are split into tokens/subword tokens

The tokenizer converts review text into numbers that BERT can understand.

In [11]:
model = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model)


def tokenizer_function(text):
    return tokenizer(
        text["reviewText"],
        padding=True,
        truncation=True,
        max_length=128
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Tokenise all datasets

This applies the BERT tokenizer to the train, validation, and test sets.

Each review becomes:
- `input_ids`: token IDs
- `attention_mask`: tells BERT which tokens are real and which are padding
- `label`: the target class

In [12]:
tokenized_train_set = train_set.map(tokenizer_function, batched=True)
tokenized_test_set = test_set.map(tokenizer_function, batched=True)
tokenized_val_set = val_set.map(tokenizer_function, batched=True)

print(tokenized_train_set[0])

Map:   0%|          | 0/11469 [00:00<?, ? examples/s]

Map:   0%|          | 0/1434 [00:00<?, ? examples/s]

Map:   0%|          | 0/1434 [00:00<?, ? examples/s]

{'reviewText': 'Pros -Solid builtGood sound qualityFits well in the earBetter calling qualityCons -You get Static shocks sometimes\n', 'label': 2, 'input_ids': [101, 4013, 2015, 1011, 5024, 2328, 24146, 2614, 3737, 8873, 3215, 2092, 1999, 1996, 4540, 20915, 3334, 4214, 3737, 8663, 2015, 1011, 2017, 2131, 10763, 28215, 2823, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

### Set dataset format for PyTorch

This tells the Hugging Face datasets to return PyTorch tensors for the columns needed by BERT.

In [13]:
tokenized_train_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

tokenized_test_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

tokenized_val_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

### Explore the tokenizer
These next cells are not part of training. They are just to help understand what BERT tokenization is doing.

In [14]:
sample_text = train_set["reviewText"][0]
sample_tokens = tokenizer(sample_text, padding=True, truncation=True, max_length=128)

print("Original text (first 200 chars):\n", sample_text[:200])
print("\nNumber of input_ids:", len(sample_tokens["input_ids"]))
print("First 20 token IDs:", sample_tokens["input_ids"][:20])
print("First 20 decoded tokens:", tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"][:20]))
print("Attention mask (first 30 values):", sample_tokens["attention_mask"][:30])

Original text (first 200 chars):
 Pros -Solid builtGood sound qualityFits well in the earBetter calling qualityCons -You get Static shocks sometimes


Number of input_ids: 28
First 20 token IDs: [101, 4013, 2015, 1011, 5024, 2328, 24146, 2614, 3737, 8873, 3215, 2092, 1999, 1996, 4540, 20915, 3334, 4214, 3737, 8663]
First 20 decoded tokens: ['[CLS]', 'pro', '##s', '-', 'solid', 'built', '##good', 'sound', 'quality', '##fi', '##ts', 'well', 'in', 'the', 'ear', '##bet', '##ter', 'calling', 'quality', '##con']
Attention mask (first 30 values): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### Observe subword tokenization

In [15]:
test_words = ["Worked", "Gadget", "Disappoints"]

for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:20s} → {tokens}")

Worked               → ['worked']
Gadget               → ['ga', '##dget']
Disappoints          → ['di', '##sa', '##pp', '##oint', '##s']


### Evaluation Before fine-tuning
We evaluate the pre-trained BERT model without training it on our dataset to establish a baseline performance.

#### Create baseline BERT model and metric function

This cell creates a BERT sequence classification model with 3 output labels.

The `compute_metrics` function calculates:
- accuracy
- precision
- recall
- F1-score

These are the same types of metrics the other models should use so comparison is fair.

In [16]:
id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

baseline_model = BertForSequenceClassification.from_pretrained(
    model,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Reusable evaluation function

This function runs validation and test evaluation for any BERT-style model.

It also appends results into the `results` list so the notebook can later create a comparison table.

In [17]:
results = []

def run_val_and_test(model, model_name, tokenized_val_set, tokenized_test_set, compute_metrics, output_dir):
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=output_dir,
            report_to="none",
            per_device_train_batch_size=16,
        ),
        compute_metrics=compute_metrics
    )

    # Validation
    val_results = trainer.evaluate(eval_dataset=tokenized_val_set)

    print(f"\nValidation metrics {model_name}")
    print("Accuracy:", val_results["eval_accuracy"])
    print("Precision:", val_results["eval_precision"])
    print("Recall:", val_results["eval_recall"])
    print("F1:", val_results["eval_f1"])

    # Test
    test_results = trainer.predict(tokenized_test_set)

    print(f"\nTest metrics {model_name}")
    print("Accuracy:", test_results.metrics["test_accuracy"])
    print("Precision:", test_results.metrics["test_precision"])
    print("Recall:", test_results.metrics["test_recall"])
    print("F1:", test_results.metrics["test_f1"])

    # Store the results
    results.append({
        "model": model_name,
        "val_accuracy": val_results["eval_accuracy"],
        "val_precision": val_results["eval_precision"],
        "val_recall": val_results["eval_recall"],
        "val_f1": val_results["eval_f1"],
        "test_accuracy": test_results.metrics["test_accuracy"],
        "test_precision": test_results.metrics["test_precision"],
        "test_recall": test_results.metrics["test_recall"],
        "test_f1": test_results.metrics["test_f1"]
    })

    return val_results, test_results


#### Run BERT before fine-tuning

This evaluates the baseline BERT model on the validation and test sets before any training happens.

In [18]:
pre_val_results, pre_test_results = run_val_and_test(
    model=baseline_model,
    model_name="before_fine_tuning",
    tokenized_val_set=tokenized_val_set,
    tokenized_test_set=tokenized_test_set,
    compute_metrics=compute_metrics,
    output_dir="/content/drive/MyDrive/NLP/Assignment3/output",
)


Validation metrics before_fine_tuning
Accuracy: 0.6562064156206415
Precision: 0.4306068599016902
Recall: 0.6562064156206415
F1: 0.5199917786097041

Test metrics before_fine_tuning
Accuracy: 0.6562064156206415
Precision: 0.6691830638827168
Recall: 0.6562064156206415
F1: 0.5207151477742608


### Fine-Tuning BERT
The pre-trained BERT model is fine-tuned on the training dataset to adapt it to the sentiment classification task.

#### Create a fresh BERT model for full fine-tuning
This reloads `bert-base-uncased` as a 3-class classification model.

In [19]:
full_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


##### Train the fine-tuned BERT model

This is where actual BERT training happens.

Important settings:
- `num_train_epochs=3`: trains for 3 passes over the training data
- `learning_rate=2e-5`: small learning rate commonly used for BERT
- `eval_strategy=epoch`: evaluates after each epoch
- `load_best_model_at_end=True`: keeps the best validation model

In [20]:
full_trainer = Trainer(
    model=full_model,
    args=TrainingArguments(
        output_dir="/content/drive/MyDrive/NLP/Assignment3/output/bert_full_finetune",
        report_to="none",
        logging_dir="/content/drive/MyDrive/NLP/Assignment3",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        logging_steps=1000
    ),
    train_dataset=tokenized_train_set,
    eval_dataset=tokenized_val_set,
    compute_metrics=compute_metrics
)

full_trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.418441,0.842399,0.820937,0.842399,0.819615
2,0.452843,0.437699,0.850767,0.833241,0.850767,0.839349
3,0.298908,0.485161,0.847978,0.838901,0.847978,0.842925


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2151, training_loss=0.36780273033152733, metrics={'train_runtime': 814.7259, 'train_samples_per_second': 42.231, 'train_steps_per_second': 2.64, 'total_flos': 2263235840941824.0, 'train_loss': 0.36780273033152733, 'epoch': 3.0})

#### Evaluate fine-tuned BERT

This evaluates the trained BERT model on validation and test data and stores the results.

In [21]:
full_val_results, full_test_results = run_val_and_test(
    model=full_trainer.model,
    model_name="full_fine_tuning",
    tokenized_val_set=tokenized_val_set,
    tokenized_test_set=tokenized_test_set,
    compute_metrics=compute_metrics,
    output_dir="/content/drive/MyDrive/NLP/Assignment3/output/bert_full_results",
)


Validation metrics full_fine_tuning
Accuracy: 0.8507670850767085
Precision: 0.8332412682088569
Recall: 0.8507670850767085
F1: 0.8393490526368331

Test metrics full_fine_tuning
Accuracy: 0.8333333333333334
Precision: 0.8112708517677854
Recall: 0.8333333333333334
F1: 0.819409547573038


## Results table

This converts the stored model results into a dataframe.

Right now, this table only includes the BERT methods, we will soon implement all the rest of the methods 

In [22]:
results_df = pd.DataFrame(results)
print(results_df)

                model  val_accuracy  val_precision  val_recall    val_f1  \
0  before_fine_tuning      0.656206       0.430607    0.656206  0.519992   
1    full_fine_tuning      0.850767       0.833241    0.850767  0.839349   

   test_accuracy  test_precision  test_recall   test_f1  
0       0.656206        0.669183     0.656206  0.520715  
1       0.833333        0.811271     0.833333  0.819410  
